In [25]:
import pandas as pd
import requests
import json
import pandas as pd
from tqdm import tqdm

In [26]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2:latest"

In [27]:

# df = pd.read_csv("mozilla_core_clean.csv")
# Rename by specific mapping
# df = df.rename(columns={"Unnamed: 0": "id"})
# eval_docs = df.sample(400, random_state=42)

In [28]:
# eval_docs.head()

In [29]:
# eval_docs.to_csv("random_samples.csv", index=False)

In [30]:
eval_docs = pd.read_csv("random_samples.csv")

In [ ]:
def generate_queries(bug_text):
    prompt = f"""
You are generating evaluation queries for a retrieval system.

Given the following bug report, generate EXACTLY 2 realistic developer search queries.

Return output strictly in JSON format like this:
{{
  "queries": [
    "query 1",
    "query 2"
  ]
}}

BUG REPORT:
\"\"\"{bug_text}\"\"\"
"""

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.3
        }
    }

    response = requests.post(OLLAMA_URL, json=payload)
    result = response.json()["response"]
    # result = None
    # print(response.status_code)
    # print(response.text)
    # Extract JSON safely
    try:
        json_start = result.find("{")
        json_end = result.rfind("}") + 1
        parsed = json.loads(result[json_start:json_end])
        return parsed["queries"]
    except:
        return []

In [32]:
evaluation_data = []

for _, row in tqdm(eval_docs.iterrows(), total=len(eval_docs)):
    bug_text = row["Description"]
    doc_id = row["id"]

    queries = generate_queries(bug_text)

    for q in queries:
        evaluation_data.append({
            "query": q,
            "relevant_doc_id": doc_id
        })

eval_df = pd.DataFrame(evaluation_data)
eval_df.to_csv("evaluation_queries.csv", index=False)

print("Saved evaluation_queries.csv")

  0%|          | 1/400 [00:03<20:09,  3.03s/it]

500
{"error":"model requires more system memory (2.9 GiB) than is available (816.4 MiB)"}


  0%|          | 2/400 [00:05<17:53,  2.70s/it]

500
{"error":"model requires more system memory (2.9 GiB) than is available (816.9 MiB)"}


  1%|          | 3/400 [00:07<15:43,  2.38s/it]

...

500
{"error":"model requires more system memory (2.3 GiB) than is available (485.2 MiB)"}


  2%|▏         | 8/400 [00:19<15:51,  2.43s/it]

500
{"error":"model requires more system memory (2.3 GiB) than is available (522.5 MiB)"}


  2%|▏         | 8/400 [00:21<17:34,  2.69s/it]


KeyboardInterrupt: 